<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex12.1-power-grid-stability-estimation/Ex12.1_01_wls_baseline_light.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*
*This is the **light** version: every TODO is written out, and you only replace the `...` marked lines with what the comment beside them says.*


<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_12.1 · Notebook 01 — Weighted Least Squares, the Baseline

**Paired with L12.1 · Power Grid Stability Estimation**

**Prerequisite: notebook 00.**

Before anything is learned, run the estimator every utility already runs.
Weighted least squares has been in production since the 1970s, it is fast, it
is understood, and it comes with a residual you can reason about.

Any new method has to explain what it adds. *"It uses a neural network"* is not
an answer, and the numbers you produce here are what everything later is
measured against — so keep them.

### What WLS does

Weight each measurement residual by how much you trust that meter, and minimise

$$\hat{x} = \arg\min_x \sum_{j=1}^{m}
\frac{\left(z_j - h_j(x)\right)^2}{\sigma_j^2}$$

Solved by Gauss-Newton: linearise $h$, solve the normal equations, update,
repeat.

Its weakness is not accuracy. It is that it needs enough well-placed
measurements, and where it does not have them it leans on assumptions.

---

## 0 · Setup

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex12.1-power-grid-stability-estimation/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# --- setup: every Part 2 notebook opens with this cell ------------------
# Needs course_core.py, pinn_core.py and problem.py beside this notebook.
# On Colab the files cell above fetched them from the public course repository.
import os
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from pinn_core import *                                  # noqa: F401,F403
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt

set_seed(88)
print("device:", DEVICE, " dtype:", torch.get_default_dtype())

In [ ]:
ref = pb.load("00_reference")
V_true, th_true = ref["V"], ref["th"]
Y = pb.build_ybus()
print(f"reference state loaded: {len(V_true)} buses")

## 1 · Generate measurements from the truth

We know the true state, so we can manufacture realistic readings from it: apply
the measurement function and add noise at each meter's stated accuracy. This is
the only reason we can score anything — in a control room there is no truth to
compare against.

In [ ]:
ms = pb.default_measurements()
rng = np.random.default_rng(3)
z = pb.synth_measurements(V_true, th_true, Y, ms, rng)

print(f"  {'#':>3}{'kind':>6}{'bus':>5}{'sigma':>9}{'reading':>12}")
for k, ((kind, bus), s, val) in enumerate(zip(ms.spec, ms.sigma, z)):
    print(f"  {k:>3}{kind:>6}{bus:>5}{s:>9.4f}{val:>12.4f}")

**Expected output**

> Fourteen rows. Voltage magnitudes near 1.0, angles small and negative,
> injections matching the bus table from notebook 00 to within the noise.

## 2 · Run the estimator

`pb.wls_estimate` returns the state plus an `info` dict carrying the objective
and the **normalised residuals**, which are the standard bad-data diagnostic.

Note the `pb.` on `pb.error_table` below, and keep it. `course_core` exports an
`error_table` of its own that formats a Markdown table, and the setup cell has
already pulled that one into the notebook's namespace. The one you want here is
the per-bus estimation report, and it lives in `problem.py`.

In [ ]:
V_wls, th_wls, info = pb.wls_estimate(z, ms, Y)
print(f"converged: {info['converged']}   iterations: {info['iterations']}   "
      f"objective J = {info['J']:.2f}")

res_default = pb.error_table(V_true, th_true, V_wls, th_wls,
                             ms.measured_buses(), label="WLS, default set")
fig = pb.plot_network_state(V_wls, th_wls, V_true, th_true,
                            ms.measured_buses(), "WLS, default measurement set")
plt.show()

**Expected output**

> Converged in about 5 iterations, objective **J around 12**.
>
> Worst |V| error about **6.7e-4** at metered buses and **6.4e-4** at the single
> unmetered bus. With a healthy, redundant measurement set WLS is accurate — that
> is the control, and it is why the comparison later is interesting rather than
> rigged.

## 3 · Now take the meters away

The thin set has one PMU and two injection measurements: rank 4 of the 10
unknowns. WLS has no way to determine the rest from data, and the `ridge` term
inside the solver is the only thing keeping it from failing outright.

Watch what happens. This is the motivation for notebook 02.

In [ ]:
ms_thin = pb.thin_measurements()
z_thin = pb.synth_measurements(V_true, th_true, Y, ms_thin,
                               np.random.default_rng(3))
V_t, th_t, info_t = pb.wls_estimate(z_thin, ms_thin, Y)

print(f"converged: {info_t['converged']}   iterations: {info_t['iterations']}")
res_thin = pb.error_table(V_true, th_true, V_t, th_t,
                          ms_thin.measured_buses(), label="WLS, thin set")
pb.comparison_table([("WLS default", res_default), ("WLS thin", res_thin)])
fig = pb.plot_network_state(V_t, th_t, V_true, th_true,
                            ms_thin.measured_buses(), "WLS, thin measurement set")
plt.show()

**Expected output**

> Worst |V| error jumps to around **2.3e-2** at metered buses and **3.0e-2** at
> unmetered ones — roughly **45 times worse** than with the full set, and the
> degradation is worst exactly where there are no meters.
>
> The estimator does not announce this. It converges and returns numbers.

## TODO 1 — bad data

Inject a gross error into one measurement and find it using the normalised
residuals. A well-designed estimator should point at the bad meter, not smear
the error across the whole state.

Use `pb.synth_measurements(..., bad=(index, offset))` with an offset of a few
tenths — far larger than the meter's sigma.

In [ ]:
# TODO 1 --- a gross error, and whether the residuals find it -----------------------------------------
# Two `...` to replace:
#   line 1  ->  (4, 0.15)                                       corrupt measurement 4 by 0.15 (try other indices and sizes)
#   line 2  ->  int(np.nanargmax(info_b["normalised_residuals"]))   the measurement with the largest normalised residual
z_bad = pb.synth_measurements(V_true, th_true, Y, ms, np.random.default_rng(3), bad=...)   # <- (4, 0.15)
V_b, th_b, info_b = pb.wls_estimate(z_bad, ms, Y)
k = ...                                           # <- int(np.nanargmax(info_b["normalised_residuals"]))
print(f"corrupted measurement 4 ({ms.spec[4]});  largest normalised residual at {k} ({ms.spec[k]})")
print("normalised residuals:", np.round(info_b["normalised_residuals"], 2))
# Answer below, in a markdown cell: did it land on the corrupted one? If not, why not?
# How large does the error have to be before it is detectable? (change the 0.15 and rerun)
# ------------------------------------------------------------------------------

## TODO 2 — where would you put one more meter?

The thin set has two metered buses. Add **one** more measurement — your choice
of kind and bus — and find the placement that most reduces the worst-bus error.

Six buses is small enough to try every option, so do that rather than guessing,
and say in your report that the search was exhaustive.

In [ ]:
# TODO 2 --- where would one more meter help most? --------------------------------------------------
# Two `...` to replace, inside the loops:
#   line 1  ->  list(pb.thin_measurements().spec) + [(kind, bus)]            the thin set plus one candidate meter
#   line 2  ->  float(np.abs(V_try - V_true).max())                           worst |V| error across ALL buses
ranking = []
for kind in ("V", "TH", "P", "Q"):
    for bus in range(pb.N_BUS):
        spec = ...                                # <- list(pb.thin_measurements().spec) + [(kind, bus)]
        ms_try = pb.MeasurementSet(spec)
        z_try = pb.synth_measurements(V_true, th_true, Y, ms_try, np.random.default_rng(3))
        V_try, th_try, _ = pb.wls_estimate(z_try, ms_try, Y)
        worst = ...                               # <- float(np.abs(V_try - V_true).max())
        ranking.append((worst, kind, bus))
ranking.sort()
print("best five placements (worst |V| error, kind, bus):")
for worst, kind, bus in ranking[:5]:
    print(f"  {worst:.2e}   {kind:2s} at bus {bus}")
print("\nIs the best one the bus you would have guessed? It is usually not the busiest bus.")
# ------------------------------------------------------------------------------

In [ ]:
pb.save("01_wls", V_wls=V_wls, th_wls=th_wls,
        V_thin=V_t, th_thin=th_t,
        J=np.array([info["J"]]))
print("\nnotebook 01 complete — go to 02_algebraic_pinn")